# Prophet - SAFEX white maize with external regressors

Prophet models a price series as *trend + seasonality + holidays*. For South African white maize this
notebook:

* uses `prophet` (the package formerly called `fbprophet`)
* uses **South African public holidays** (`ZA`) instead of US ones
* switches **daily seasonality off** - the data has one price per day, so there is no intraday pattern
  to learn - and keeps weekly, monthly and **yearly (harvest cycle)** seasonality
* adds the external drivers (**ZAR/USD, rainfall, fuel price**) as extra regressors
* evaluates on the same chronological 80 / 20 split as the other notebooks

## 1. Imports

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# `fbprophet` was renamed to `prophet`  (pip install prophet)
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.utilities import regressor_coefficients

warnings.filterwarnings('ignore')
%matplotlib inline

## 2. Load and align the datasets

Same loading logic as in the SARIMAX notebook: SAFEX prices + external drivers, inner join,
business-day calendar with forward-fill.

In [ ]:
# ---- Data settings ---------------------------------------------------------
PRICES_FILE = 'safex_prices.csv'   # Date, Close (R/ton)  [+ optional Open, High, Low, Volume]
EXOG_FILE   = 'sa_factors.csv'     # Date, ZAR_USD, Rainfall_mm, Fuel_Price
TARGET_COL  = 'Close'              # SAFEX white maize price column
EXOG_COLS   = ['ZAR_USD', 'Rainfall_mm', 'Fuel_Price']   # external drivers
TRAIN_FRAC  = 0.8                  # chronological split: first 80% train, last 20% test

# ---- Prophet settings --------------------------------------------------------
CV_INITIAL = '730 days'    # first training window for cross-validation (shrink it if your history is short)
CV_PERIOD  = '90 days'     # spacing between cut-offs
CV_HORIZON = '90 days'     # how far ahead each cross-validation forecast looks
N_AHEAD    = 20            # business days for the forward forecast at the end

In [ ]:
# Load the SAFEX white maize prices and the South African external drivers.
# (If your dates are written dd/mm/yyyy, add dayfirst=True to both read_csv calls.)
prices = pd.read_csv(PRICES_FILE, parse_dates=['Date'], index_col='Date')
exog_data = pd.read_csv(EXOG_FILE, parse_dates=['Date'], index_col='Date')

# Tidy up: ascending dates and no duplicated dates
prices = prices.sort_index()
prices = prices[~prices.index.duplicated(keep='last')]
exog_data = exog_data.sort_index()
exog_data = exog_data[~exog_data.index.duplicated(keep='last')]

# Fail early, with a readable message, if a column name does not match the settings above
missing = [c for c in [TARGET_COL] if c not in prices.columns] + \
          [c for c in EXOG_COLS if c not in exog_data.columns]
if missing:
    raise KeyError(f'Column(s) not found in the CSV files: {missing}')

In [ ]:
# Safeguard: put lower-frequency drivers (e.g. monthly fuel prices) on a daily calendar BEFORE the
# join, carrying the last known value forward. Otherwise a monthly series only matches ~12 dates a
# year, and the inner join below would silently throw away almost all of the daily price rows.
# (.last().ffill() also fills blanks if all drivers sit in one wide file with empty cells.)
exog_data = exog_data.resample('D').last().ffill()

# Merge both dataframes with an inner join so that the dates align
data = prices.join(exog_data, how='inner')

# Business-day calendar; forward-fill weekend / holiday gaps
data = data.resample('B').ffill()
data = data.dropna(subset=[TARGET_COL] + EXOG_COLS)

In [ ]:
print(f'{len(data)} business days   {data.index.min().date()} -> {data.index.max().date()}')
data[[TARGET_COL] + EXOG_COLS].describe().round(2)

## 3. Prophet data format and train / test split

Prophet expects the columns `ds` (date) and `y` (target); every extra regressor is an extra column.

In [ ]:
prophet_df = pd.DataFrame({'ds': data.index, 'y': data[TARGET_COL].to_numpy()})
for col in EXOG_COLS:
    prophet_df[col] = data[col].to_numpy()

train_size = int(len(prophet_df) * TRAIN_FRAC)
train_df = prophet_df.iloc[:train_size].copy()
test_df = prophet_df.iloc[train_size:].copy()

print(f'train: {len(train_df)} days  ({train_df.ds.min().date()} -> {train_df.ds.max().date()})')
print(f'test : {len(test_df)} days  ({test_df.ds.min().date()} -> {test_df.ds.max().date()})')
prophet_df.head()

## 4. Fit Prophet

`build_prophet()` returns a fresh, configured model so that the same settings are used for the model with
regressors, the one without, and the final refit.

In [ ]:
def build_prophet(use_regressors=True):
    m = Prophet(yearly_seasonality=True,      # harvest / planting cycle
                weekly_seasonality=True,
                daily_seasonality=False,      # one price per day -> no intraday seasonality
                changepoint_prior_scale=0.05)
    m.add_country_holidays(country_name='ZA')                         # South African public holidays
    m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    if use_regressors:
        for col in EXOG_COLS:
            m.add_regressor(col)                                      # ZAR/USD, rainfall, fuel price
    return m


m = build_prophet(use_regressors=True)
m.fit(train_df)

# Predict for every date (history + test block); the test rows use the test block's regressor values
forecast = m.predict(prophet_df[['ds'] + EXOG_COLS])
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

> **Read the test score with care.** Like the SARIMAX notebook, the regressor values for the test block are
> the *actual* ones. That tells you how well the price follows the drivers *if the drivers are known*; it is
> not a pure forecast. The "no regressors" model below is the fair benchmark.

In [ ]:
def score(y_true, y_pred):
    """MAE (R/ton), RMSE (R/ton) and MAPE (%)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'MAPE %': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)}


results = {}
fc_test = forecast.iloc[train_size:]
results['Prophet + drivers'] = score(test_df['y'], fc_test['yhat'])

# Benchmark 1: Prophet without the external regressors
m0 = build_prophet(use_regressors=False)
m0.fit(train_df[['ds', 'y']])
fc0 = m0.predict(prophet_df[['ds']]).iloc[train_size:]
results['Prophet, no drivers'] = score(test_df['y'], fc0['yhat'])

# Benchmark 2: repeat the last training price
results['Naive: last training price'] = score(test_df['y'], np.full(len(test_df), train_df['y'].iloc[-1]))

pd.DataFrame(results).T.round(2).sort_values('MAE')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_df['ds'], train_df['y'], label='training')
plt.plot(test_df['ds'], test_df['y'], label='actual')
plt.plot(fc_test['ds'], fc_test['yhat'], label='Prophet forecast')
plt.fill_between(fc_test['ds'], fc_test['yhat_lower'], fc_test['yhat_upper'], color='k', alpha=.15)
plt.title('Prophet with external regressors: forecast vs actuals')
plt.ylabel('R / ton')
plt.legend(loc='upper left', fontsize=8)
plt.show()

In [ ]:
# Trend, seasonalities, holidays and the effect of each regressor
fig = m.plot_components(forecast)
plt.show()

# Size and direction of each regressor's effect (in price units per unit of the regressor)
regressor_coefficients(m)

## 5. Cross-validation (rolling origin)

Prophet's own time-series cross-validation: it repeatedly trains on an expanding window and forecasts
the following `CV_HORIZON`. It only touches the **training block**, so the test block stays untouched.

In [ ]:
df_cv = cross_validation(m, initial=CV_INITIAL, period=CV_PERIOD, horizon=CV_HORIZON)

df_p = performance_metrics(df_cv)                       # error by forecast horizon
df_p.head(5)

In [ ]:
performance_metrics(df_cv, rolling_window=1)            # one overall row

plot_cross_validation_metric(df_cv, metric='mae')
plt.show()

## 6. Forward forecast

Refit on all available history, then forecast the next `N_AHEAD` business days. Future regressor values are
unknown, so the last observed values are carried forward - **replace them with your own scenarios**
(e.g. a weaker rand) to see how sensitive the forecast is.

In [ ]:
m_full = build_prophet(use_regressors=True)
m_full.fit(prophet_df)

future = pd.DataFrame({'ds': pd.bdate_range(prophet_df['ds'].iloc[-1] + pd.offsets.BDay(1),
                                            periods=N_AHEAD)})
for col in EXOG_COLS:
    future[col] = prophet_df[col].iloc[-1]          # scenario: drivers stay where they are
# Example: rand 5% weaker ->  future['ZAR_USD'] *= 1.05

future_fc = m_full.predict(future)
future_fc[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].round(2)